code incomplete

In [2]:
!pip install torch torchvision

In [3]:
import torch
import torchvision
import torch.nn.functional as F  # Parameterless functions, like (some) activation functions
import torchvision.datasets as datasets  # Standard datasets
import torchvision.transforms as transforms  # Transformations we can perform on our dataset for augmentation
from torch import optim  # For optimizers like SGD, Adam, etc.
import torch.nn as nn  # All neural network modules
from torch.utils.data import DataLoader
from tqdm import tqdm  # For nice progress bar!
import matplotlib.pyplot as plt

Setting the device
-

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

Pretrained Model
-

VGG16 has already learned to identify edges, textures, and shapes from millions of images in ImageNet

In [17]:
model = torchvision.models.vgg16(weights="DEFAULT") 

Hyper-Parameters
-

In [18]:
# Updated Hyper-parameters
num_classes = 10    
learning_rate = 1e-3
num_epochs = 3
batch_size =64

Freezing the weights
-

In [19]:
for param in model.parameters():
    param.requires_grad=False
model.avgpool = nn.Identity()
model.classifier=nn.Sequential(
    nn.Linear(512,100), nn.ReLU(),nn.Linear(100,50),nn.ReLU(),nn.Linear(50,num_classes)
)
model.to(device)

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

Load the Data
-

In [20]:
train_dataset = datasets.CIFAR10(
    root="dataset/", train=True, transform=transforms.ToTensor(), download=True
)
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)

In [21]:
type(train_dataset)

torchvision.datasets.cifar.CIFAR10

Loss and optimizer
-

In [22]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

Train Network
--

In [23]:
for epoch in range(num_epochs):
    losses =[]
    for batch_idx, (data, targets) in enumerate(tqdm(train_loader)):
        # Get data to cuda if possible
        data = data.to(device=device)
        targets = targets.to(device=device)

        # forward
        scores = model(data)
        loss = criterion(scores, targets)
        losses.append(loss.item())
        # backward
        optimizer.zero_grad()
        loss.backward()

        # gradient descent or adam step
        optimizer.step()
        
    print(f"Cost at epoch {epoch} is {sum(losses)/len(losses):.5f}")

100%|██████████| 782/782 [00:13<00:00, 56.02it/s]


Cost at epoch 0 is 1.26721


100%|██████████| 782/782 [00:13<00:00, 55.94it/s]


Cost at epoch 1 is 1.09926


100%|██████████| 782/782 [00:14<00:00, 55.58it/s]

Cost at epoch 2 is 1.04536


##### Check accuracy on training & test to see how good our model

In [24]:
def check_accuracy(loader, model):
    num_correct = 0
    num_samples = 0
    model.eval()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device)
            y = y.to(device=device)

            scores = model(x)
            _, predictions = scores.max(1)
            num_correct += (predictions == y).sum()
            num_samples += predictions.size(0)
        
        print(f"accuracy = {float(num_correct)/float(num_samples)*100:.2f}")

    model.train()


In [25]:
test_dataset = datasets.CIFAR10(
    root="dataset/", train=False, transform=transforms.ToTensor(), download=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

In [26]:
print(f"Accuracy on training set: {check_accuracy(train_loader, model)*100:.2f}")
print(f"Accuracy on test set: {check_accuracy(test_loader, model)*100:.2f}")

accuracy = 65.41


TypeError: unsupported operand type(s) for *: 'NoneType' and 'int'